# Test matrizActividades.py


In [ ]:

"""Notebook para probar las funcionalidades y pulir el desempeño de la función que genera la matriz de actividades."""

## 1. Configuración global

In [1]:

import os
import sys
import time
from pathlib import Path
import pandas as pd
import re
import json
import pickle
import logging
import pymongo
from pymongo.errors import ConnectionFailure
from deltalake import DeltaTable, write_deltalake
from pprint import pprint
from datetime import datetime

from eerssa.secret import Keys

logging.basicConfig(level=logging.INFO)


# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


# Ubicación del directorio DELTA LAKE TABLE
table_path = "./test/deltalake_2025"

# DELTA LAKE Connection

# Verify the existence of the DELTA LAKE table
if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    print(f"Conectado a la tabla Delta Lake en: {table_path}")


Success!!!


INFO:root::::: Conexion exitosa con MongoDB ::::


Conectado a la tabla Delta Lake en: ./test/deltalake_2025


In [10]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import gestionOT as OrdenTrabajo                   # Convert from PDF_ot to obj_ot
from eerssa import matrizActividades as Actividades     # process ot.data["actividades"]

In [35]:
reload( OrdenTrabajo )
reload( Actividades  )

<module 'eerssa.matrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/matrizActividades.py'>

## Obtener Ordenes de trabajo desde MongoDB

In [26]:
test_json = CurrentCollection.find_one({"id_ot":150217}) 

In [27]:
type(test_json)

dict

In [28]:
test_obj = OrdenTrabajo.GestionOt.from_dict(test_json)

activ = test_obj.data['actividades']
df_test = pd.DataFrame( activ)
df_test


,Item,Actividad,Evento,Ali,Alimentador,Tipo,InicioEvento,FinEvento
0,1,None,DAÑO REPORTADO POR EL CENTRO DE CONTROL,None,None,None,None,None
1,2,TRANSP,Nos trasladamos desde la agencia EERSSA Zamora...,None,None,TRANSPORTE,2025-03-30 20:03:00,2025-03-30 20:30:00
2,3,NO PROG,"Tunantza Alto, daño informado por el Sr. Juan ...",ALI,Zamora II,CORRECTIVO,2025-03-30 20:30:00,2025-03-30 21:06:00
3,4,None,FECHA / HORA DESCONEXIÓN: 30-Mar-2025 / 16:00\...,None,None,None,None,None
4,5,TRANSP,Nos trasladamos desde Tunantza Alto a la agenc...,None,None,TRANSPORTE,2025-03-30 21:06:00,2025-03-30 21:28:00
5,6,None,None,None,None,None,None,None
6,7,None,SE LABORA: RS - LL de 20:03 a 21:28,None,None,None,None,None
7,8,None,MATERIALES OCUPADOS:\n_1 Tirafusible de 6 Amp....,None,None,None,None,None


In [37]:
test_matriz = Actividades.ConvertirOT_a_ActividadesCSV(test_obj)
test_matriz

/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
0,1,informativa,DAÑO REPORTADO POR EL CENTRO DE CONTROL,INFO,·,No,No,No,·,·,...,2025-03-30 00:00:01,2025-03-30 00:00:02,0,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Tunantza Alto,150217,Orden de trabajo Zamora 30-03-2025 (RS).pdf
1,2,transporte,Nos trasladamos desde la agencia EERSSA Zamora...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2025-03-30 20:03:00,2025-03-30 20:30:00,27,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Tunantza Alto,150217,Orden de trabajo Zamora 30-03-2025 (RS).pdf
2,3,REDES,"Tunantza Alto, daño informado por el Sr. Juan ...",NO PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2025-03-30 20:30:00,2025-03-30 21:06:00,36,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Tunantza Alto,150217,Orden de trabajo Zamora 30-03-2025 (RS).pdf
4,5,transporte,Nos trasladamos desde Tunantza Alto a la agenc...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2025-03-30 21:06:00,2025-03-30 21:28:00,22,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Tunantza Alto,150217,Orden de trabajo Zamora 30-03-2025 (RS).pdf
5,7,se_labora,SE LABORA: RS - LL de 20:03 a 21:28\rMATERIALE...,LABORA,·,No,No,No,·,·,...,2025-03-30 00:00:01,2025-03-30 00:00:02,0,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Tunantza Alto,150217,Orden de trabajo Zamora 30-03-2025 (RS).pdf
